# Recalculating Coastal Upwelling Transport Index (CUTI), Jacox et al., (2018)

For CUTI we need the Ekman net contribution ($Ek_{int}$) + the geostrophic part ($U_{geo}$)

* For the $Ek_{int}$ we integrate the boundaries at every 1 degree perimeter, let the equations be as follow:

\
\begin{aligned}
Q^{(E)}_{E} &= \int_{y_1}^{y_2} U_E(x_E,y)\,dy, = 0 \\
Q^{(W)}_{E} &= - \int_{y_1}^{y_2} U_E(x_W,y)\,dy, \\
Q^{(N)}_{E} &= \int_{x_W}^{x_E} V_E(x,y_2)\,dx, \\
Q^{(S)}_{E} &= - \int_{x_W}^{x_E} V_E(x,y_1)\,dx, \\
Q_{\mathrm{div}}
&= Q^{(E)}_{E} + Q^{(W)}_{E} + Q^{(N)}_{E} + Q^{(S)}_{E}
= \iint\limits_{A} \nabla \!\cdot\! \mathbf{M}_E \, dA, \\
\bar{w}_E
&= \frac{Q_{\mathrm{div}}}{A}
= \frac{1}{\rho\,f}\, \bigl(\nabla \times \boldsymbol{\tau}\bigr)_z .
\end{aligned}

* For geostrophic transport ($U_{geo}$), $\Delta \mathrm{SSH}$ is the difference between the coastal SSH values at the northernmost and southernmost grid points within the 1° segment. The $MLD$ is defined as the width of the Rossby radius of deformation (30 km; R1).

$$
\begin{aligned}
u_{\mathrm{geo}} &= \frac{g}{f}\,\frac{\Delta \mathrm{SSH}}{d_{\mathrm{coast}}}, \\[1ex]
\mathbf{U}_{\mathrm{geo}} &= \mathbf{u}_{\mathrm{geo}} \times \mathrm{MLD}, \\[1ex]
\text{CUTI} &= U_{\text{ek}} + U_{\text{geo}} \\
&= \frac{\nabla \tau}{\rho_w f} \cdot \hat{k} - \frac{gD}{f} \frac{\partial \eta}{\partial y}.
\end{aligned}
$$

where $g$ is the gravitational acceleration, $D$ is the Ekman layer depth, and $d$ is distance to the coast.

In [1]:
import xarray as xr
import matplotlib.pyplot as plt
from cmocean import cm # for oceanography-specific colormaps
from scipy.io import loadmat
import numpy as np
import re
from xml.etree import ElementTree as ET
from matplotlib.path import Path
import pandas as pd
from matplotlib.colors import TwoSlopeNorm 
import matplotlib.colors as mcolors
from tqdm import tqdm 

## Functions

In [2]:
time = pd.date_range(start="1980-01",end="2016-01",freq='M')

ds_ = xr.open_dataset('../../../NHCS/hincast_1980-2015/croco_avg_Y1980M01.nc', 
                     chunks = {'time':1})

mask_rho = ds_.mask_rho.values 
mask_nan = np.where(mask_rho, 1, np.nan)
expanded_mask = np.tile(mask_nan[np.newaxis, :, :], (432, 1, 1))

/tmp/ipykernel_380834/70548169.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  time = pd.date_range(start="1980-01",end="2016-01",freq='M')


In [3]:
## kml to struct
def kml2struct(kml_file):
    """
    Import a .kml file as a list of dictionary structures with fields:
    Geometry, Name, Description, Lon, Lat, and BoundingBox.
    """
    # Read the KML file
    try:
        with open(kml_file, 'r', encoding='utf-8') as file:
            txt = file.read()
    except Exception as e:
        raise FileNotFoundError(f"Unable to open file {kml_file}: {e}")

    # Regular expression to match Placemark tags
    expr = r"<Placemark.+?>.+?</Placemark>"
    object_strings = re.findall(expr, txt, re.DOTALL)
    
    kml_struct = []

    for obj_str in object_strings:
        # Extract Name
        name_match = re.search(r"<name.*?>(.*?)</name>", obj_str, re.DOTALL)
        name = name_match.group(1).strip() if name_match else "undefined"

        # Extract Description
        desc_match = re.search(r"<description.*?>(.*?)</description>", obj_str, re.DOTALL)
        desc = desc_match.group(1).strip() if desc_match else ""

        # Determine Geometry Type
        geometry = ""
        if "<Point" in obj_str:
            geometry = "Point"
        elif "<LineString" in obj_str:
            geometry = "Line"
        elif "<Polygon" in obj_str:
            geometry = "Polygon"

        # Extract Coordinates
        coord_match = re.search(r"<coordinates.*?>(.*?)</coordinates>", obj_str, re.DOTALL)
        if not coord_match:
            continue  # Skip if no coordinates are found
        coord_str = coord_match.group(1).strip()
        coord_list = np.array([list(map(float, coord.split(','))) for coord in coord_str.split()])

        # Separate Lon, Lat, and handle Polygons
        lon = coord_list[:, 0]
        lat = coord_list[:, 1]
        if geometry == "Polygon":
            # Close the polygon by appending NaN
            lon = np.append(lon, np.nan)
            lat = np.append(lat, np.nan)

        # Create BoundingBox
        bounding_box = [[lon.min(), lat.min()], [lon.max(), lat.max()]]

        # Append to kml_struct
        kml_struct.append({
            "Geometry": geometry,
            "Name": name,
            "Description": desc,
            "Lon": lon,
            "Lat": lat,
            "BoundingBox": bounding_box
        })

    return kml_struct

In [4]:
arch_kml_zona1 = "/gxfs_work/geomar/smomw662/NHCS/hindcast/CROCO_BioEBUS_1990-2010/indices/BK111km.kml"
R1 = kml2struct(arch_kml_zona1)

# Extract Lon and Lat from the first polygon
lonb1 = R1[0]["Lon"]
latb1 = R1[0]["Lat"]

LON, LAT = np.meshgrid(ds_.lon_rho.compute().values[0,:],ds_.lat_rho.compute().values[:,0])
# Create a path from the polygon coordinates
polygon = Path(np.column_stack((lonb1, latb1)))

# Flatten the LON and LAT to create coordinate pairs
lonlat_points = np.column_stack((LON.ravel(), LAT.ravel()))

# Check which points are inside the polygon
mask = polygon.contains_points(lonlat_points).reshape(LON.shape)

# Convert the mask to NaN for the outshore region
inshore_mask = np.where(mask, 1, np.nan)

mask_rho = ds_.mask_rho.values 
mask_nan = np.where(mask_rho, 1, np.nan)

### $U^{Ek}$ calculation 

In [5]:
def qdiv_for_band(Uek, Vek,
                  lat_south, lat_north,   # e.g. -5, -4  (south < north)
                  lon_west=-83, lon_east=-70,
                  delta_y=9250, delta_x=9250, N=12):
    """
    Compute Qdiv for a given lat band [lat_south, lat_north]
    using your current method.
    """

    # slice: note lat_south < lat_north
    Uek_ = (Uek).sel(
        lat=slice(lat_south, lat_north), lon=slice(lon_west, lon_east)
    )
    Vek_ = (Vek).sel(
        lat=slice(lat_south, lat_north), lon=slice(lon_west, lon_east)
    )

    # Boolean mask of valid ocean cells
    valid = xr.where(np.isfinite(Uek_), 1, 0)

    valid_e = valid.shift(lon=-1)  # east neighbour
    valid_w = valid.shift(lon=+1)  # west neighbour

    west_mask = (valid == 1) & (valid_w == 0)
    east_mask = (valid == 1) & (valid_e == 0)

    easternmost = Uek_.where(east_mask)   # inshore
    westernmost = Uek_.where(west_mask)   # offshore

    southernmost = Vek_.min(dim='lat', skipna=True)
    northernmost = Vek_.max(dim='lat', skipna=True)

    # QE, QW, QN, QS exactly as you wrote
    QE = (easternmost.mean('lon')
          .rolling(lat=2, center=False)
          .mean(skipna=True).sum(dim='lat') * delta_y) ## as this value points to the coastal boundary it should be 0

    QW = -(westernmost.mean('lon')
           .rolling(lat=2, center=False)
           .mean(skipna=True).sum(dim='lat') * delta_y) ## points to the left

    QN = (northernmost
          .rolling(lon=2, center=False)
          .mean(skipna=True).sum(dim='lon') * delta_y)

    QS = -(southernmost
           .rolling(lon=2, center=False)
           .mean(skipna=True).sum(dim='lon') * delta_y) #points south
    
    Qdiv = (QW + QS + QN) / (delta_x * (N - 1))

    # Qdiv = (QE + QW + QS + QN) / (delta_x * (N - 1))
    # Qdiv_Sv = (QE + QW + QS + QN) /10e6

    return Qdiv

## Data

In [6]:
import glob

files = sorted(glob.glob('/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y*M*_uwnd_vwnd.nc'))
print(len(files), files[:5], files[-5:])

# Peek at first two
for fi in files[:3]:
    with xr.open_dataset(fi) as ds:
        print(fi, ds['bulk_time'].load().values[:3], ds['bulk_time'].load().values[-3:])

432 ['/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y1980M10_uwnd_vwnd.nc', '/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y1980M11_uwnd_vwnd.nc', '/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y1980M12_uwnd_vwnd.nc', '/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y1980M1_uwnd_vwnd.nc', '/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y1980M2_uwnd_vwnd.nc'] ['/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y2015M5_uwnd_vwnd.nc', '/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y2015M6_uwnd_vwnd.nc', '/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y2015M7_uwnd_vwnd.nc', '/gxfs_work/geomar/smomw662/NHCS/Winds_input/Winds_6h/extracted_winds/croco_blk_ERA5_Y2015M8_uwnd_vwnd.nc', '/gxfs_work/geomar/

In [7]:
from tqdm import tqdm

alpha = 112.27
rho_w = 1025.0
lat_top_start = -5   # first northern edge
lat_top_end   = -16  # last northern edge

for fi in tqdm(files):
    fn = fi[-21:-12]
    ## Open the data 
    ds = xr.open_dataset(fi, chunks=-1)
    ds = ds[['uwnd', 'vwnd']]
    ds = ds.load()
    
    origin = np.datetime64('1980-01-01T00:00:00')  # example; choose your true origin
    abs_time = origin + ds.bulk_time
    
    #################
    #### Panda dates
    pdt = pd.DatetimeIndex(abs_time.values.astype('datetime64[ns]'))
    year  = xr.DataArray(pdt.year,  coords=ds.bulk_time.coords, dims=ds.bulk_time.dims, name='year')
    month = xr.DataArray(pdt.month, coords=ds.bulk_time.coords, dims=ds.bulk_time.dims, name='month')
    day   = xr.DataArray(pdt.day,   coords=ds.bulk_time.coords, dims=ds.bulk_time.dims, name='day')
    hour   = xr.DataArray(pdt.hour,   coords=ds.bulk_time.coords, dims=ds.bulk_time.dims, name='hour')
    
    ##################
    ## Assign coords
    ds = ds.assign_coords(
        year = (('bulk_time',), year.values),
        month = (('bulk_time',), month.values),
        day = (('bulk_time',), day.values),
        hour = (('bulk_time',), hour.values),
    )
    
    time = xr.DataArray(abs_time.astype('datetime64[ns]'),
                        coords=ds.bulk_time.coords, dims=ds.bulk_time.dims, name='time')
    
    # Add it as a coordinate and keep bulk_time as-is
    ds = ds.assign_coords(time=('bulk_time', time.values))
    
    #########################
    ### Pump calculation####
    # A: Earth radius (meters)
    A = 6371000.0
    
    # Ensure uwnd and vwnd are in the expected order and dims (time, eta_u, xi_u), (time, eta_v, xi_v)
    u = xr.DataArray(
        ds['uwnd'].data,
        dims=('time','eta_u','xi_u'),
        coords=dict(
            time=ds.time.data,
            lat_u=(('eta_u','xi_u'), ds_.lat_u.compute().values),
            lon_u=(('eta_u','xi_u'), ds_.lon_u.compute().values),
        ),
        name='u10'
    )
    
    v = xr.DataArray(
        ds['vwnd'].data,
        dims=('time','eta_v','xi_v'),
        coords=dict(
            time=ds.time.data,
            lat_v=(('eta_v','xi_v'), ds_.lat_v.compute().values),
            lon_v=(('eta_v','xi_v'), ds_.lon_v.compute().values),
        ),
        name='v10'
    )
    
    # ----------------------------
    # Compute wind speed at faces with proper alignment
    # ----------------------------
    # Map v to u points: average in xi (x) direction -> loses 1 cell in xi
    v_on_u = 0.5 * (
        v.isel(xi_v=slice(0, -1)).rename({'eta_v':'eta_u', 'xi_v':'xi_u'}) +
        v.isel(xi_v=slice(1,  None)).rename({'eta_v':'eta_u', 'xi_v':'xi_u'})
    )
    
    # Align shapes by trimming to common interior
    eta_u_size = min(u.sizes['eta_u'], v_on_u.sizes['eta_u'])
    xi_u_size  = min(u.sizes['xi_u'],  v_on_u.sizes['xi_u'])
    
    u_c      = u.isel(eta_u=slice(0, eta_u_size), xi_u=slice(0, xi_u_size))
    v_on_u_c = v_on_u.isel(eta_u=slice(0, eta_u_size), xi_u=slice(0, xi_u_size))
    
    # Map u to v points: average in eta (y) direction -> loses 1 cell in eta
    u_on_v = 0.5 * (
        u.isel(eta_u=slice(0, -1)).rename({'eta_u':'eta_v', 'xi_u':'xi_v'}) +
        u.isel(eta_u=slice(1,  None)).rename({'eta_u':'eta_v', 'xi_u':'xi_v'})
    )
    
    # Align shapes
    eta_v_size = min(v.sizes['eta_v'], u_on_v.sizes['eta_v'])
    xi_v_size  = min(v.sizes['xi_v'],  u_on_v.sizes['xi_v'])
    
    v_c      = v.isel(eta_v=slice(0, eta_v_size), xi_v=slice(0, xi_v_size))
    u_on_v_c = u_on_v.isel(eta_v=slice(0, eta_v_size), xi_v=slice(0, xi_v_size))
    
    # Wind speed magnitudes at u and v faces
    ws_u = xr.apply_ufunc(np.hypot, u_c, v_on_u_c,dask='allowed')  # (time, eta_u_size, xi_u_size)
    ws_v = xr.apply_ufunc(np.hypot, u_on_v_c, v_c,dask='allowed')  # (time, eta_v_size, xi_v_size)
    
    # ----------------------------
    # Surface wind stresses at faces
    # ----------------------------
    rho_air = 1.22
    Cd = 1.3e-3
    taux_u = rho_air * Cd * ws_u * u_c
    tauy_v = rho_air * Cd * ws_v * v_c
    
    # ----------------------------
    # Average stresses to rho points
    # ----------------------------
    # tau_x from u -> rho (average in xi)
    taux_rho = 0.5 * (
        taux_u.isel(xi_u=slice(0, -1)).rename({'eta_u':'eta_rho','xi_u':'xi_rho'}) +
        taux_u.isel(xi_u=slice(1,  None)).rename({'eta_u':'eta_rho','xi_u':'xi_rho'})
    ).drop_vars(['lat_u','lat_v'])
    
    # tau_y from v -> rho (average in eta)
    tauy_rho = 0.5 * (
        tauy_v.isel(eta_v=slice(0, -1)).rename({'eta_v':'eta_rho','xi_v':'xi_rho'}) +
        tauy_v.isel(eta_v=slice(1,  None)).rename({'eta_v':'eta_rho','xi_v':'xi_rho'})
    ).drop_vars(['lon_u','lon_v'])
    
    # Align tau arrays to common rho grid size
    eta_rho_size = min(taux_rho.sizes['eta_rho'], tauy_rho.sizes['eta_rho'])
    xi_rho_size  = min(taux_rho.sizes['xi_rho'],  tauy_rho.sizes['xi_rho'])
    
    taux_rho = taux_rho.isel(eta_rho=slice(0, eta_rho_size), xi_rho=slice(0, xi_rho_size))
    tauy_rho = tauy_rho.isel(eta_rho=slice(0, eta_rho_size), xi_rho=slice(0, xi_rho_size))
    
    # Attach 2D rho coords (trimmed to match)
    lat_rho = xr.DataArray(
        ds_.lat_rho.compute().values[:eta_rho_size, :xi_rho_size], 
        dims=('eta_rho','xi_rho'), 
        name='lat'
    )
    lon_rho = xr.DataArray(
        ds_.lon_rho.compute().values[:eta_rho_size, :xi_rho_size], 
        dims=('eta_rho','xi_rho'), 
        name='lon'
    )
    taux_rho = taux_rho.assign_coords(lat=lat_rho, lon=lon_rho)
    tauy_rho = tauy_rho.assign_coords(lat=lat_rho, lon=lon_rho)

            # Ensure same sizes for curl computation
    eta_c = min(tauy_rho.sizes['eta_rho'], tauy_rho.sizes['eta_rho'])
    xi_c  = min(tauy_rho.sizes['xi_rho'],  tauy_rho.sizes['xi_rho'])
    
    Omega = 7.2921e-5
    f = 2.0 * Omega * np.sin(np.deg2rad(lat_rho))
    # Avoid divide by zero near equator
    f = xr.where(np.abs(f) < 1e-10, np.nan, f)

    ### Uek
    Uek = tauy_rho / (rho_w * f)
    Vek = taux_rho / (rho_w * f)

    Uek = Uek * inshore_mask[:eta_c, :xi_c]* mask_nan[:eta_c, :xi_c]
    Vek = Vek * inshore_mask[:eta_c, :xi_c]* mask_nan[:eta_c, :xi_c]
    
    band = (lat_rho >= -16) & (lat_rho <= -5)
    
    Uek_ = Uek.where(band)
    Vek_ = Vek.where(band)

    tau = xr.Dataset(
    data_vars={
        'Uek': (('time','lat','lon'),Uek_.data),
        'Vek': (('time','lat','lon'),Vek_.data),
    },
    coords={
        'lat': lat_rho.data[:,0],
        'lon': lon_rho.data[0,:],
    },)

    ### Net Ekman contribution (Ek_int or Uek)
    results = []
    lat_band_centers = []
    
    for lat_north in range(lat_top_start, lat_top_end - 1, -1):  # -4, -5, ..., -16
        lat_south = lat_north - 1                                # -5, -6, ..., -17
    
        Qdiv_band = qdiv_for_band(
            tau.Uek, tau.Vek,
            lat_south=lat_south, lat_north=lat_north,
            lon_west=-83, lon_east=-70,
            delta_y=9250, delta_x=9250, N=12
        )
    
        results.append(Qdiv_band)
        lat_band_centers.append(0.5 * (lat_south + lat_north))
        
    Qdiv_all = xr.concat(results, dim='lat_band')
    
    Ek_int = Qdiv_all.assign_coords(lat_band=lat_band_centers)
    Ek_int['time'] = Uek_.time

    #### Exporting
    Ek_int.to_dataset(name='Uek').drop_encoding().to_netcdf(f'CUTI/Ek_int{fn}.nc')

100%|████████████████████████████████████████████████████████████████████████████████| 432/432 [1:40:49<00:00, 14.00s/it]


$
u_{\mathrm{geo}} = \frac{g}{f}\,\frac{\Delta \mathrm{SSH}}{d_{\mathrm{coast}}}
$

$
\mathbf{U}_{\mathrm{geo}} = \mathbf{u}_{\mathrm{geo}} \times \mathrm{MLD}
$

#### SSH

In [8]:
SSH = loadmat("../../NHCS/Processed/SST_SSH.mat")

time = pd.date_range(start="1980-01",end="2016-01",freq='M')
ds_SSH = xr.Dataset(
    data_vars={
        'SSH':(("time", "lat","lon"), SSH['SSH'].transpose(2, 1, 0)*expanded_mask),
        
    },
    coords = {
        'lon':ds.lon_rho.compute().values[0,:],
        'lat':ds.lat_rho.compute().values[:,0],
        'time':time
}
)
ds_SSH

FileNotFoundError: [Errno 2] No such file or directory: '../../NHCS/Processed/SST_SSH.mat'

In [ ]:
SSHc_ = (ds_SSH.SSH*inshore_mask)

In [ ]:
SSHc_.isel(time=0).plot(robust=True)

In [ ]:
SSH_ = SSHc_.sel(lat=slice(-17,-5),lon=slice(-83,-70))

lat_top_start = -5   # first northern edge
lat_top_end   = -16  # last northern edge

delta_SSH = []
# results_Sv = []

for lat_north in tqdm(range(lat_top_start, lat_top_end - 1, -1)):  # -4, -5, ..., -16
    lat_south = lat_north - 1                                # -5, -6, ..., -17

    # 1° band slice
    SSH_band = SSH_.sel(lat=slice(lat_south, lat_north))

    southernmost = SSH_band.min(dim='lat', skipna=True)
    northernmost = SSH_band.max(dim='lat', skipna=True)
    
    delta_SSH.append(northernmost-southernmost)

ssh = xr.concat(delta_SSH, dim='lat_band')
ssh = ssh.assign_coords(
    lat_band=np.arange(lat_top_start - 0.5, lat_top_end - 0.5 - 1, -1)
)


In [ ]:
ssh

In [ ]:
ssh.mean(dim='lon').plot.contourf(robust=True)

In [ ]:
lat_top_start = -5   # first northern edge
lat_top_end   = -16  # last northern edge

delta_f = []
# results_Sv = []

for lat_north in tqdm(range(lat_top_start, lat_top_end - 1, -1)):  # -4, -5, ..., -16
    lat_south = lat_north - 1                                # -5, -6, ..., -17

    # 1° band slice
    f_band = f.sel(lat=slice(lat_south, lat_north))

    southernmost = f_band.min(dim='lat', skipna=True)
    northernmost = f_band.max(dim='lat', skipna=True)
    
    delta_f.append((northernmost+southernmost)/2)

f_ = xr.concat(delta_f, dim='lat_band')
f_ = f_.assign_coords(
    lat_band=np.arange(lat_top_start - 0.5, lat_top_end - 0.5 - 1, -1)
)

In [ ]:
d_coast = 111000#1 grado de latitud en m
g = 9.81

u_geo = (g/f_)*(ssh/d_coast)
#u_geo = u_geo.mean(dim='lon')

#u_geo.plot.contourf(robust=True)

seasonality(u_geo.mean('lon').rename('m/s')).plot.contourf()

#### MLD

In [ ]:
MLD = loadmat("../../NHCS/Processed/WMLD_MLD.mat")

MLD_ds = xr.Dataset(
    data_vars={
        'MLD': (("time", "lat","lon"), MLD['MLD'].transpose(2, 1, 0)*expanded_mask* inshore_mask),
    },
    coords = {
        'lon':ds.lon_rho.compute().values[0,:],
        'lat':ds.lat_rho.compute().values[:,0],
        'time':time
}
)

MLD_ds

In [ ]:
mld = (MLD_ds*mld_30km_mask).MLD
mld.isel(time=0).plot()

In [ ]:
lat_top_start = -5   # first northern edge
lat_top_end   = -16  # last northern edge

delta_mld = []
# results_Sv = []

for lat_north in tqdm(range(lat_top_start, lat_top_end - 1, -1)):  # -4, -5, ..., -16
    lat_south = lat_north - 1                                # -5, -6, ..., -17

    # 1° band slice
    mld_band = mld.sel(lat=slice(lat_south, lat_north))

    southernmost = mld_band.min(dim='lat', skipna=True)
    northernmost = mld_band.max(dim='lat', skipna=True)
    
    delta_mld.append(northernmost-southernmost)

mld_ = xr.concat(delta_mld, dim='lat_band')
mld_ = mld_.assign_coords(
    lat_band=np.arange(lat_top_start - 0.5, lat_top_end - 0.5 - 1, -1)
)

In [ ]:
#mld_.mean(dim='lon',skipna=True).plot.contourf(robust=True)
seasonality(mld_.mean(dim=('lon')).rename('m')).plot.contourf()

#### $U^{geo}$

In [ ]:
U_geo = (u_geo * mld_)
seasonality((u_geo * mld_).mean(dim='lon')).rename('m2/s').plot.contourf(robust=True)
plt.title('$U^{Geo}$')

The Ugeo points offshore and 

In [ ]:
CUTI = U_ek + (U_geo)#m2/s

seasonality(CUTI.mean(dim='lon')).rename('$m^2s^{-1}$').plot.contourf(robust=True)

In [ ]:
seasonality(CUTI.mean(dim='lon')).mean(dim='lat_band').rename('$m^2s^{-1}$').plot()
plt.grid()

### Jacox et al., 2018 like plot

In [ ]:
U_ek.mean(dim='time').plot(y='lat_band',figsize=(3,7),label='$U^{Ek}$')
(U_geo).mean(dim=('lon','time')).plot(y='lat_band',label='$U^{Geo}$')
plt.grid()
plt.xlabel('$m^2s^{-1}$')
plt.legend()

In [ ]:
normalize(seasonality(U_ek).mean('lat_band')).plot(label='$U^{Ek}$')
normalize(seasonality(U_geo * -1).mean(dim=('lat_band','lon'))).plot(label='$U^{Geo}$')
normalize(seasonality(CUTI.mean(dim='lon')).mean('lat_band')).plot(label='$CUTI$')
plt.legend()
plt.grid()
plt.title('CUTI components')

In [ ]:
# 1) Build monthly series (as before) and align months
def to_month_1_12(da):
    if 'month' in da.coords:
        m = da['month'].values
        if np.min(m) == 0 and np.max(m) == 11:
            da = da.assign_coords(month=(('month',), m + 1))
        return da
    if 'time' in da.coords:
        return (da.assign_coords(month=da['time'].dt.month)
                  .swap_dims({'time': 'month'})
                  .groupby('month').mean())
    raise ValueError("DataArray needs a 'month' or 'time' coordinate")

months_labels = ['J','F','M','A','M','J','J','A','S','O','N','D']
full_months = np.arange(1, 13)

UGEO = seasonality(U_geo*-1).mean(dim=('lat_band','lon'))
EKMN = seasonality(U_ek).mean(dim=('lat_band'))

UGEO = to_month_1_12(UGEO).reindex(month=full_months)
EKMN = to_month_1_12(EKMN).reindex(month=full_months)

u = UGEO.values
e = EKMN.values

# 2) Compute total and percentage contributions
total = u + e
eps = 1e-12  # to avoid division by zero
pct_u = 100 * u / np.where(np.abs(total) < eps, np.nan, total)
pct_e = 100 * e / np.where(np.abs(total) < eps, np.nan, total)

# Optional: handle months with zero/near-zero total (set to 0 instead of NaN)
pct_u = np.where(np.isnan(pct_u), 0.0, pct_u)
pct_e = np.where(np.isnan(pct_e), 0.0, pct_e)

# 3) Print a quick table
for m, pu, pe, tt in zip(full_months, pct_u, pct_e, total):
    print(f"Month {m:2d}: UGEO {pu:6.1f}% | Ekman {pe:6.1f}% | Total={tt:.3g}")

# 4) Plot stacked percent bar chart (100% stacked)
fig, ax = plt.subplots(figsize=(9, 4.8))
x = full_months
width = 0.8

# Split positive/negative totals if you prefer sign-aware; here we show simple 100% stack
ax.bar(x, pct_u, width=width, label='UGEO (−)', color='steelblue', alpha=0.9)
ax.bar(x, pct_e, width=width, bottom=pct_u, label='Ekman', color='orange', alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(months_labels, fontsize=11)
ax.set_xlabel('Month')
ax.set_ylabel('Contribution (%)')
ax.set_title('Monthly % contribution to CUTI (UGEO vs Ekman), 16°S to 5°S')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)
ax.legend(frameon=False, ncol=2, loc='upper left')

plt.tight_layout()
plt.show()

#### Validation with $W_{mld}$

In [ ]:
WMLD_data = loadmat("../../NHCS/Processed/WMLD_MLD.mat")
WMLD_ds = xr.Dataset(
    data_vars={
        'WMLD': (("time", "lat","lon"), WMLD_data['WMLD'].transpose(2, 0, 1)*expanded_mask*inshore_mask),
    },
    coords = {
        'lon':ds.lon_rho.compute().values[0,:],
        'lat':ds.lat_rho.compute().values[:,0],
        'time':time
}
)

WMLD_ds

In [ ]:
## Sverdrup
A = 6371.0e3

ndelta_y = (np.deg2rad(WMLD_ds.lat.shift(lat=-1) - WMLD_ds.lat) * A)
ndelta_x = (
    np.deg2rad(WMLD_ds.lon.shift(lon=-1) - WMLD_ds.lon) * A
    * np.cos(np.deg2rad(WMLD_ds.lat))
)

da = ndelta_x * ndelta_y

#---- Wmld
Sv_wmld = WMLD_ds.WMLD * da
Wmld_Sv = Sv_wmld.sel(lat=slice(-16,-5)).sum(['lon','lat']) / 1e6 #SV



In [ ]:
def lat_band_mean(
    f,
    lat_top_start=-5,
    lat_top_end=-16,
):
    """
    Compute (f_north + f_south)/2 for successive 1° latitude bands.

    Parameters
    ----------
    f : xarray.DataArray
        Must have a 'lat' dimension (and optionally time/lon/etc.).
    lat_top_start : float or int
        Northern latitude of the first band (e.g. -5).
    lat_top_end : float or int
        Northern latitude of the last band (e.g. -16).

    Returns
    -------
    f_band : xarray.DataArray
        delta_f for each 1° band, with new dimension 'lat_band'.
        Any other original dimensions (e.g. time) are preserved.
    """

    delta_f = []
    lat_band_centers = []

    for lat_north in tqdm(range(lat_top_start, lat_top_end - 1, -1)):
        lat_south = lat_north - 1

        # 1° band slice
        f_band = f.sel(lat=slice(lat_south, lat_north))

        # skip bands with no points
        if f_band.sizes.get("lat", 0) == 0:
            continue

        southernmost = f_band.min(dim="lat", skipna=True)
        northernmost = f_band.max(dim="lat", skipna=True)

        delta_f.append((northernmost + southernmost) / 2)
        lat_band_centers.append(0.5 * (lat_south + lat_north))

    # concat over bands
    f_out = xr.concat(delta_f, dim="lat_band")
    f_out = f_out.assign_coords(lat_band=np.array(lat_band_centers))
    f_out.name = "delta_f"

    return f_out

In [ ]:
mask_=ds.mask_rho.where(ds.mask_rho!=0,np.nan).to_numpy()

dy_ = lat_band_mean(ndelta_y, lat_top_start=-5, lat_top_end=-16)

In [ ]:
WMLD_ds.WMLD.sel(lat=slice(-16,-5)).isel(time=0).plot()

In [ ]:
CUTI.shape

In [ ]:
#---- CUTI
CUTI_Sv = CUTI.mean('lon') * dy_ 
# CUTI_Sv = CUTI_Sv.sum(['lat_band'])
seasonality(CUTI_Sv.sum(['lat_band'])/1e6).rename('Sv').plot()

In [ ]:
#---- w
w_Sv = W.mean('lon') * dy_ 
# CUTI_Sv = CUTI_Sv.sum(['lat_band'])
seasonality(CUTI_Sv.sum(['lat_band'])/1e6).rename('Sv').plot()

In [ ]:
(seasonality(((WMLD_ds.WMLD*ndelta_y).mean('lon').sel(lat=slice(-16,-5)) * 111e3).sum('lat'))/1e6).plot()

In [ ]:
seasonality(Wmld_Sv).plot()

### CUTI and $W_{mld}$ comparison

In [ ]:
(WMLD_ds.WMLD.sel(lat=slice(-16,-5))*111e3).mean(dim=('lon','lat')).plot(figsize=(10,5),label='$W$')
CUTI.mean(dim=('lon','lat_band')).rename('m2/s').plot(label='CUTI')
plt.grid()
plt.legend()